In [2]:
from pyCicero import preprocess
from pyCicero import run_cicero

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.io import mmread

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

In [89]:
names_mapping = {
    "intron":"Intron",
    "promoter-TSS":"Promoter-TSS",
    "exon":"Exon",
    "3'":"3' UTR",
    "5'":"5' UTR",
}
def map_detailed_annotations(x):
    if "|" in x:
        if "SINE" in x:
            return "SINE"
        if "LINE" in x:
            return "LINE"
        if "LTR" in x:
            return "LTR"
        return "Other Repeats"
        # return annotation.split(" ")[0]
    if "CpG" in x:
        return "CpG Island"
    x = x.split(" ")[0]
    return names_mapping.get(x, x)

In [ ]:
counts = mmread("../../05_2025_raw_data/ATAC/05_02_2025_neuronal_FMatrix.mtx").T.tocsr()

In [121]:
obs_names = pd.read_csv("../../05_2025_raw_data/ATAC/05_02_2025_neuronal_FMatrix_Col_Names.csv", index_col=0)
vars = pd.read_csv("../../05_2025_raw_data/ATAC/05_02_2025_neuronal_FMatrix_Row_Names.csv", index_col=0)
vars.index = ["merged-id" + str(x) for x in vars.index.values]
obs = pd.read_csv("../../05_2025_raw_data/ATAC/05_02_2025_Meta_Data.csv", index_col=0)
obs = obs.reindex(obs_names["x"])

In [111]:
adata = sc.AnnData(counts, obs = obs, var = vars)

In [112]:
adata = adata[:,["chr" in x for x in vars["x"].values]]

In [125]:
annotations_neuronal_atac = pd.read_csv("../../05_2025_Merged_ATAC_Neuronal_Homer_annotations.txt", sep = "\t")
annotations_neuronal_atac["Detailed Annotation Simplified"] = list(map(map_detailed_annotations, annotations_neuronal_atac["Detailed Annotation"].values))
annotated_vars = annotations_neuronal_atac.set_index(annotations_neuronal_atac.columns[0]).reindex(adata.var.index)

In [ ]:
adata.var = annotated_vars
adata.var.insert(3, "Mean", (adata.var["Start"] + adata.var["End"])/2) #need to have a column named mean 

In [127]:
adata = preprocess.preprocess_cicero(adata)

preprocess.py: 2025-05-05 00:46:27 WARNING  Counts layers not found. Coppying X to counts
preprocess.py: 2025-05-05 00:46:28 INFO     Estimaing Size Factors and Normalizing Data
preprocess.py: 2025-05-05 00:46:44 INFO     Finished Size Factors and Normallization storing normalized sparse matrix into 'data'
preprocess.py: 2025-05-05 00:46:44 INFO     Running TF-IDF
preprocess.py: 2025-05-05 00:46:48 INFO     Finsihed TF-IDF
preprocess.py: 2025-05-05 00:46:48 INFO     Running TruncatedSVD
preprocess.py: 2025-05-05 00:49:43 INFO     Finsihed TruncatedSVD
preprocess.py: 2025-05-05 00:49:43 INFO     Running UMAP
preprocess.py: 2025-05-05 00:49:44 INFO     Finsihed UMAP


In [128]:
cicero_adata = preprocess.make_cicero_adata(adata, k = 50)

preprocess.py: 2025-05-05 00:50:22 INFO     Using adata OBSM X_umap to agregate cells
preprocess.py: 2025-05-05 00:50:22 INFO     Calculating overlap
preprocess.py: 2025-05-05 00:50:22 INFO     Generating Pseudobulk cicero observations with seed: 0 and k: 50
preprocess.py: 2025-05-05 00:50:39 INFO     Reached Maximum itterations in pseudobulk observation generation. Consider increasing 'max_itterations'
preprocess.py: 2025-05-05 00:50:39 INFO     Found 3251 good choices
preprocess.py: 2025-05-05 00:50:39 INFO     Finished calculating overlap
preprocess.py: 2025-05-05 00:50:39 INFO     Aggregating Cells
preprocess.py: 2025-05-05 00:51:14 INFO     Finished Aggregating Cells


In [192]:
cons = run_cicero.run_cicero(cicero_adata)

run_cicero.py: 2025-05-05 04:35:51 INFO     Generating Windows
run_cicero.py: 2025-05-05 04:35:51 INFO     Starting distance_parameter_estimation
100%|██████████| 500/500 [00:44<00:00, 11.17it/s]
run_cicero.py: 2025-05-05 04:36:45 INFO     Starting multiprocessing pool for estimate_distance_parameter_parallel with 96 processes
run_cicero.py: 2025-05-05 04:38:29 INFO     Finished distance_parameter_estimation
run_cicero.py: 2025-05-05 04:38:29 INFO     Starting generate_cicero_models
run_cicero.py: 2025-05-05 04:40:16 INFO     Starting Cicero with 96 processes
run_cicero.py: 2025-05-05 04:41:22 INFO     Finished generate_cicero_models
run_cicero.py: 2025-05-05 04:41:24 INFO     Starting assemble_connections
run_cicero.py: 2025-05-05 04:42:02 INFO     Finished assemble_connections


In [194]:
cons[np.abs(cons["coaccess_score"]) > 0.2].head()

,Peak1,Peak2,coaccess_score
195,merged-id1000,merged-id1001,0.337160
268,merged-id1000,merged-id999,0.265556
479,merged-id100000,merged-id99999,0.587284
833,merged-id100003,merged-id99987,0.202139
2087,merged-id100014,merged-id100015,0.312871
